In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Load original dataset
df = pd.read_csv(
    "../Data/CosciaRios2012_DataBase.csv"
)

# Load geographical boundaries
gdf = gpd.read_file(
    "../Data/Boundaries/Municipios_2010_5.shp"
)

print(df.shape)
print(gdf.shape)

print(df.columns.tolist())
print(gdf.columns.tolist())

In [ ]:
df["Code"] = df["Code"].astype(str).str.zfill(5)

gdf["Code"] = (
    gdf["CVE_ENT"].astype(str).str.zfill(2)
    + gdf["CVE_MUN"].astype(str).str.zfill(3)
)

In [ ]:
gdf["Code"] = gdf["CVE_ENT"] + gdf["CVE_MUN"]

print("Boundary codes:", gdf["Code"].nunique())
print("CSV codes:", df["Code"].nunique())

missing_from_map = set(df["Code"].astype(str)) - set(gdf["Code"])
missing_from_csv = set(gdf["Code"]) - set(df["Code"].astype(str))

print("CSV municipalities without boundary:", missing_from_map)
print("Boundaries without CSV municipality:", missing_from_csv)

In [ ]:
# Inspect data types
print(df.dtypes)

# Number of unique states
print("States:", df["State"].nunique())

# Number of unique years
print("Years:", df["Year"].nunique())

# Missing values
print(df.isnull().sum())

In [ ]:
# Check duplicate municipality-year combinations
duplicates = df.duplicated(
    subset=["Code", "Year"]
).sum()

print("Duplicate observations:", duplicates)

# Check year range
print("First year:", df["Year"].min())
print("Last year:", df["Year"].max())

# Check cartel indicators
cartel_columns = [
    "Beltran_Leyva",
    "Beltran_Leyva_Family",
    "Familia",
    "Golfo",
    "Juarez",
    "Sinaloa",
    "Sinaloa_Family",
    "Tijuana",
    "Zetas",
    "Otros"
]

for col in cartel_columns:
    print(col, df[col].unique())

In [ ]:
state_year = (
    df.groupby(["State", "Year"])[cartel_columns]
      .sum()
      .reset_index()
)

print(state_year.head())
print(state_year.shape)

In [ ]:
national_year = (
    state_year
    .groupby("Year")[cartel_columns]
    .sum()
    .reset_index()
)

print(national_year.head())

In [ ]:
state_presence = (
    state_year
    .groupby("Year")[cartel_columns]
    .apply(lambda x: (x > 0).sum())
    .reset_index()
)

print(state_presence.head())

In [ ]:
# Compare geographical breadth in 1990 and 2010

comparison = state_presence.set_index("Year").loc[
    [1990, 2010], cartel_columns
].T

comparison.columns = ["1990", "2010"]

comparison["Change"] = (
    comparison["2010"] - comparison["1990"]
)

comparison = comparison.sort_values(
    "Change", ascending=False
)

print(comparison)

In [ ]:
for cartel in cartel_columns:
    positive_years = state_presence.loc[
        state_presence[cartel] > 0, "Year"
    ]

    if len(positive_years) > 0:
        first_year = positive_years.min()
        last_year = positive_years.max()

        print(
            cartel,
            "First recorded year:", first_year,
            "Last recorded year:", last_year
        )
    else:
        print(cartel, "No recorded presence")

In [ ]:
merged = gdf.merge(
    df,
    on="Code",
    how="left"
)

print("Municipal polygons:", len(merged))
print("Unmatched CSV municipality:", "23010")

In [ ]:
df_2010 = df[df["Year"] == 2010]

map_2010 = gdf.merge(
    df_2010,
    on="Code",
    how="left"
)

print("Rows:", len(map_2010))
print("Unique municipalities:", map_2010["Code"].nunique())

In [ ]:
cartel = "Zetas"

map_2010[cartel] = map_2010[cartel].fillna(0)

fig, ax = plt.subplots(figsize=(12, 10))

map_2010.plot(
    column=cartel,
    cmap="Reds",
    legend=True,
    linewidth=0.1,
    edgecolor="gray",
    ax=ax
)

ax.set_title(f"{cartel} cartel presence in Mexican municipalities, 2010")
ax.axis("off")

plt.show()

In [ ]:
state_year = (
    df.groupby(["State", "Year"])[cartel_columns]
      .sum()
      .reset_index()
)

state_presence = (
    state_year.groupby("Year")[cartel_columns]
    .apply(lambda x: (x > 0).sum())
    .reset_index()
)

state_presence.set_index("Year")[cartel_columns].plot(
    figsize=(14, 7),
    marker="o"
)

plt.ylabel("Number of states with recorded presence")
plt.xlabel("Year")
plt.title("Recorded geographical breadth of cartel presence, 1990–2010")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
municipality_presence = (
    df.groupby("Year")[cartel_columns]
      .sum()
      .reset_index()
)

municipality_presence

In [ ]:
municipality_presence.set_index("Year")[cartel_columns].plot(
    figsize=(14, 7),
    marker="o"
)

plt.title("Recorded municipal presence of cartel indicators, 1990–2010")
plt.xlabel("Year")
plt.ylabel("Number of municipalities with recorded presence")
plt.grid(alpha=0.3)
plt.legend(title="Cartel indicator", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
annual_change = municipality_presence.set_index("Year")[cartel_columns].diff()

annual_change

In [ ]:
annual_change.max().sort_values(ascending=False)

In [ ]:
import libpysal
from libpysal.weights import Queen

#Queen-contiguity neighbours
w = Queen.from_dataframe(gdf, ids="Code")

#Number of neighbouring municipalities for each municipality
gdf["n_neighbours"] = gdf["Code"].map(w.cardinalities)

print(gdf[["Code", "NOM_MUN", "n_neighbours"]].head())

In [ ]:
# Check the neighbours of one municipality
w.neighbors["09012"]

In [ ]:
# Load municipality polygons
edges = []

for i, row1 in gdf.iterrows():
    neighbors = gdf[gdf.geometry.touches(row1.geometry)]

    for _, row2 in neighbors.iterrows():
        if row1["Code"] < row2["Code"]:
            edges.append(
                (
                    row1["Code"],
                    row2["Code"]
                )
            )

edge_df = pd.DataFrame(edges, columns=["source", "target"])
edge_df.to_csv("mexico_municipality_network.csv", index=False)

In [ ]:
import networkx as nx

# Create network
G = nx.from_pandas_edgelist(
    edge_df,
    source="source",
    target="target"
)

# Plot
plt.figure(figsize=(15, 15))

pos = nx.spring_layout(G, seed=42)

nx.draw_networkx(
    G,
    pos,
    node_size=20,
    node_color="skyblue",
    edge_color="gray",
    width=0.5,
    with_labels=False
)

plt.title("Mexican Municipality Adjacency Network")
plt.axis("off")
plt.show()

In [ ]:
import numpy as np

from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from IPython.display import HTML, display

# ----------------------------------
# 1. Create graph
# ----------------------------------

G = nx.from_pandas_edgelist(
    edge_df,
    source="source",
    target="target"
)

# Ensure identifiers match
gdf["Code"] = gdf["Code"].astype(str)
G = nx.relabel_nodes(G, str)

# ----------------------------------
# 2. Get geographic positions
# ----------------------------------

# Representative points are guaranteed to lie inside polygons
gdf["point"] = gdf.geometry.representative_point()

positions = {
    row["Code"]: (row["point"].x, row["point"].y)
    for _, row in gdf.iterrows()
}

# Keep only municipalities with valid geographic positions
nodes = [node for node in G.nodes() if node in positions]

node_positions = np.array([positions[node] for node in nodes])

node_index = {node: i for i, node in enumerate(nodes)}

edges = [
    (u, v) for u, v in G.edges()
    if u in node_index and v in node_index
]

segments = [
    [positions[u], positions[v]]
    for u, v in edges
]

edge_indices = [
    (node_index[u], node_index[v])
    for u, v in edges
]

# ----------------------------------
# 3. Plot municipality map
# ----------------------------------

fig, ax = plt.subplots(figsize=(10, 10), dpi=60)

gdf.plot(
    ax=ax,
    facecolor="white",
    edgecolor="black",
    linewidth=0.2
)

# Fix geographic limits
xmin, ymin, xmax, ymax = gdf.total_bounds

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_aspect("equal")
ax.axis("off")

ax.set_title("Mexican Municipality Adjacency Network")

# ----------------------------------
# 4. Create animation elements
# ----------------------------------

node_collection = ax.scatter(
    [],
    [],
    s=8,
    color="dodgerblue",
    zorder=3
)

edge_collection = LineCollection(
    [],
    colors="red",
    linewidths=0.6,
    alpha=0.6,
    zorder=2
)

ax.add_collection(edge_collection)

# ----------------------------------
# 5. Animate network growth
# ----------------------------------

frames = 60

def update(frame):

    n_visible = int(
        frame / (frames - 1) * len(nodes)
    )

    # Reveal municipalities
    node_collection.set_offsets(
        node_positions[:n_visible]
    )

    # Reveal connections only when both nodes exist
    visible_edges = [
        segments[i]
        for i, (u, v) in enumerate(edge_indices)
        if u < n_visible and v < n_visible
    ]

    edge_collection.set_segments(visible_edges)

    ax.set_title(
        f"Mexican Municipality Adjacency Network\n"
        f"Municipalities: {n_visible}/{len(nodes)}"
    )

    return node_collection, edge_collection

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=100,
    repeat=False
)

display(HTML(ani.to_jshtml()))

----------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------